# Qwen3-VL LoRA 학습 정리본

중복 셀과 충돌나는 `CFG` 정의를 정리한 버전입니다.  
아래 순서대로 **위에서부터 한 셀씩** 실행하면 됩니다.

In [1]:
# 처음 1회만 필요하면 실행
# 이미 설치가 끝났다면 건너뛰어도 됩니다.

# PyTorch는 RTX 5060 Ti 기준 cu128 이상이 맞습니다.
# pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128
# pip install -U bitsandbytes transformers accelerate peft trl datasets scikit-learn pandas pillow tqdm


In [2]:
from PIL import ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

import os
import random
from dataclasses import dataclass
from typing import List, Dict, Any

import numpy as np
import pandas as pd
from PIL import Image

import torch
from torch.utils.data import Dataset
from sklearn.model_selection import train_test_split

from transformers import (
    AutoProcessor,
    BitsAndBytesConfig,
    Qwen3VLForConditionalGeneration,
    TrainingArguments,
    Trainer,
)

from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
)


Skipping import of cpp extensions due to incompatible torch version 2.8.0+cu128 for torchao version 0.16.0             Please see https://github.com/pytorch/ao/issues/2919 for more info


In [3]:
from dataclasses import dataclass

@dataclass
class CFG:
    train_csv: str = "/workspace/data/train.csv"
    test_csv: str = "/workspace/data/test.csv"
    train_split_csv: str = "/workspace/data/train_split.csv"
    val_split_csv: str = "/workspace/data/val_split.csv"
    image_root = "/workspace/data/song/260401_15_2_ai_데이터배포용"
    output_dir: str = "/workspace/output"

    model_name: str = "Qwen/Qwen3-VL-8B-Instruct"
    use_4bit: bool = True

    seed: int = 42
    test_size: float = 0.2
    use_pre_split: bool = True
    max_length: int = 1024

    num_train_epochs: int = 1
    per_device_train_batch_size: int = 1
    per_device_eval_batch_size: int = 1
    gradient_accumulation_steps: int = 4
    learning_rate: float = 2e-4
    weight_decay: float = 0.01
    warmup_ratio: float = 0.03

    logging_steps: int = 10
    eval_steps: int = 100
    save_steps: int = 100

    lora_r: int = 16
    lora_alpha: int = 32
    lora_dropout: float = 0.05

    smoke_test_steps: int = -1
    dataloader_num_workers: int = 2

CFG = CFG()

In [4]:
def seed_everything(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(CFG.seed)


## 1. GPU 상태 확인

여기서 CUDA가 정상인지 먼저 확인합니다.

In [5]:
print("torch version:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("cuda version:", torch.version.cuda)
print("device count:", torch.cuda.device_count())
print("device name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

assert torch.cuda.is_available(), "CUDA GPU를 먼저 확인하세요."


torch version: 2.8.0+cu128
cuda available: True
cuda version: 12.8
device count: 1
device name: NVIDIA GeForce RTX 5060 Ti


In [6]:
def clean_text(x: Any) -> str:
    if pd.isna(x):
        return ""
    return str(x).strip()

def enrich_option(opt: str) -> str:
    opt = clean_text(opt)
    mapping = {
        "비닐": "비닐 (plastic bag)",
        "플라스틱": "플라스틱 (plastic)",
        "플라스틱 병": "플라스틱 병 (plastic bottle)",
        "유리": "유리 (glass)",
        "유리 병": "유리 병 (glass bottle)",
        "유리병": "유리병 (glass bottle)",
        "금속": "금속 (metal)",
        "금속 캔": "금속 캔 (metal can)",
        "캔": "캔 (aluminum can)",
        "종이": "종이 (paper)",
        "종이팩": "종이팩 (paper carton)",
        "박스": "박스 (cardboard box)",
        "스티로폼": "스티로폼 (styrofoam)",
    }
    return mapping.get(opt, opt)

def build_user_prompt(row: pd.Series) -> str:
    q = clean_text(row["question"])
    a = enrich_option(row["a"])
    b = enrich_option(row["b"])
    c = enrich_option(row["c"])
    d = enrich_option(row["d"])

    return f"""이미지를 보고 질문에 답하세요.

질문:
{q}

보기:
a. {a}
b. {b}
c. {c}
d. {d}

반드시 정답 문자 하나만 답하세요.
정답은 a, b, c, d 중 하나입니다."""


In [7]:
def validate_and_clean(df: pd.DataFrame, image_root: str) -> pd.DataFrame:
    df = df.copy()

    required = ["path", "question", "a", "b", "c", "d", "answer"]
    for col in required:
        df = df[df[col].notna()]

    df["question"] = df["question"].astype(str).str.strip()
    df = df[df["question"].str.len() > 0]
    df = df[df["answer"].isin(["a", "b", "c", "d"])]

    exists_mask = df["path"].apply(lambda p: os.path.exists(os.path.join(image_root, p)))
    df = df[exists_mask].reset_index(drop=True)
    return df


def drop_bad_images(df: pd.DataFrame, image_root: str, df_name: str = "df") -> pd.DataFrame:
    df = df.reset_index(drop=True).copy()
    bad_files = []

    for i, row in df.iterrows():
        image_path = os.path.join(image_root, row["path"])
        try:
            with Image.open(image_path) as img:
                img.verify()
        except Exception as e:
            bad_files.append((i, image_path, str(e)))

    print(f"{df_name} 깨진 이미지 개수:", len(bad_files))
    if bad_files:
        display(pd.DataFrame(bad_files[:20], columns=["index", "image_path", "error"]))

    bad_indices = [x[0] for x in bad_files]
    if bad_indices:
        df = df.drop(index=bad_indices).reset_index(drop=True)

    print(f"{df_name} 정리 후 개수:", len(df))
    return df


## 2. 데이터 로드

- `use_pre_split=True`면 `train_split.csv`, `val_split.csv`를 사용합니다.
- 아니면 `train.csv`에서 직접 train/val을 나눕니다.

In [8]:
if CFG.use_pre_split and os.path.exists(CFG.train_split_csv) and os.path.exists(CFG.val_split_csv):
    train_df = pd.read_csv(CFG.train_split_csv)
    val_df = pd.read_csv(CFG.val_split_csv)
    print("pre-split 파일 사용")
else:
    full_df = pd.read_csv(CFG.train_csv)
    full_df = validate_and_clean(full_df, CFG.image_root)

    train_df, val_df = train_test_split(
        full_df,
        test_size=CFG.test_size,
        random_state=CFG.seed,
        stratify=full_df["answer"],
        shuffle=True,
    )

    train_df = train_df.reset_index(drop=True)
    val_df = val_df.reset_index(drop=True)
    print("train.csv에서 직접 분할")

train_df = validate_and_clean(train_df, CFG.image_root)
val_df = validate_and_clean(val_df, CFG.image_root)

# 학습 전에 깨진 이미지 제거
train_df = drop_bad_images(train_df, CFG.image_root, "train_df")
val_df = drop_bad_images(val_df, CFG.image_root, "val_df")

print("train:", len(train_df))
print("val:", len(val_df))
print("\n=== train 분포 ===")
print(train_df["answer"].value_counts(normalize=True))
print("\n=== val 분포 ===")
print(val_df["answer"].value_counts(normalize=True))


pre-split 파일 사용
train_df 깨진 이미지 개수: 0
train_df 정리 후 개수: 2738
val_df 깨진 이미지 개수: 0
val_df 정리 후 개수: 663
train: 2738
val: 663

=== train 분포 ===
answer
b    0.257852
a    0.253835
c    0.251644
d    0.236669
Name: proportion, dtype: float64

=== val 분포 ===
answer
b    0.256410
c    0.251885
d    0.250377
a    0.241327
Name: proportion, dtype: float64


In [9]:
print("data:", os.listdir("/workspace/data"))
print("song:", os.listdir("/workspace/data/song"))

data: ['260401_15_2_ai_데이터배포용', 'song', 'test.csv', 'train.csv', 'train_split.csv', 'val_split.csv']
song: ['260401_15_2_ai_데이터배포용']


In [10]:
sample_path = os.path.join(CFG.image_root, train_df.iloc[0]["path"])
print("sample path:", sample_path)
print("exists:", os.path.exists(sample_path))


sample path: /workspace/data/song/260401_15_2_ai_데이터배포용/train/train_0952.jpg
exists: True


## 3. Dataset / Processor / Collator

In [11]:
class QwenVLDataset(Dataset):
    def __init__(self, df: pd.DataFrame, image_root: str):
        self.df = df.reset_index(drop=True)
        self.image_root = image_root

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image_path = os.path.join(self.image_root, row["path"])

        try:
            image = Image.open(image_path).convert("RGB")
        except Exception as e:
            raise RuntimeError(f"이미지 로드 실패: {image_path} | {e}")

        user_prompt = build_user_prompt(row)
        answer = clean_text(row["answer"])

        return {
            "image": image,
            "user_prompt": user_prompt,
            "answer": answer,
            "path": row["path"],
        }


In [12]:
processor = AutoProcessor.from_pretrained(CFG.model_name)
print("processor 로드 완료")


processor 로드 완료


In [13]:
class QwenVLTrainCollator:
    def __init__(self, processor):
        self.processor = processor

    def __call__(self, batch: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        texts = []
        images = []

        for item in batch:
            messages = [
                {
                    "role": "user",
                    "content": [
                        {"type": "image", "image": item["image"]},
                        {"type": "text", "text": item["user_prompt"]},
                    ],
                },
                {
                    "role": "assistant",
                    "content": [
                        {"type": "text", "text": item["answer"]},
                    ],
                },
            ]

            text = self.processor.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=False,
            )
            texts.append(text)
            images.append(item["image"])

        model_inputs = self.processor(
            text=texts,
            images=images,
            return_tensors="pt",
            padding=True,
            truncation=False,
        )

        labels = model_inputs["input_ids"].clone()
        pad_token_id = self.processor.tokenizer.pad_token_id
        if pad_token_id is not None:
            labels[labels == pad_token_id] = -100

        model_inputs["labels"] = labels
        return model_inputs

In [14]:
train_dataset = QwenVLDataset(train_df, CFG.image_root)
val_dataset = QwenVLDataset(val_df, CFG.image_root)
collator = QwenVLTrainCollator(processor)

print("train_dataset:", len(train_dataset))
print("val_dataset:", len(val_dataset))


train_dataset: 2738
val_dataset: 663


## 4. 모델 로드 + LoRA

In [15]:
quant_config = None
if CFG.use_4bit:
    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
    )

print("quant_config 준비 완료:", quant_config is not None)


quant_config 준비 완료: True


In [16]:
model = Qwen3VLForConditionalGeneration.from_pretrained(
    CFG.model_name,
    device_map="auto",
    quantization_config=quant_config,
    torch_dtype=torch.float16,
)

model.config.use_cache = False
print("모델 로드 완료")
print("model device:", next(model.parameters()).device)


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/750 [00:00<?, ?it/s]

모델 로드 완료
model device: cuda:0


In [17]:
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=CFG.lora_r,
    lora_alpha=CFG.lora_alpha,
    lora_dropout=CFG.lora_dropout,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"trainable params: {trainable:,} / {total:,}")
print(f"trainable ratio: {100 * trainable / total:.6f}%")


trainable params: 43,646,976 || all params: 8,810,770,672 || trainable%: 0.4954
trainable params: 43,646,976 / 5,052,135,664
trainable ratio: 0.863931%


## 5. Trainer 설정

처음에는 `smoke_test_steps`로 10 step만 돌려서 속도와 오류를 먼저 확인합니다.  
문제가 없으면 `max_steps=-1`로 바꾸거나 `smoke_test_steps`를 지우고 전체 학습으로 진행하세요.

In [18]:
training_args = TrainingArguments(
    output_dir=CFG.output_dir,
    num_train_epochs=CFG.num_train_epochs,
    per_device_train_batch_size=CFG.per_device_train_batch_size,
    per_device_eval_batch_size=CFG.per_device_eval_batch_size,
    gradient_accumulation_steps=CFG.gradient_accumulation_steps,
    learning_rate=CFG.learning_rate,
    weight_decay=CFG.weight_decay,
    warmup_ratio=CFG.warmup_ratio,
    logging_steps=CFG.logging_steps,
    eval_steps=CFG.eval_steps,
    save_steps=CFG.save_steps,
    eval_strategy="steps",
    save_strategy="steps",
    remove_unused_columns=False,
    report_to="none",
    fp16=True,
    dataloader_pin_memory=True,
    dataloader_num_workers=CFG.dataloader_num_workers,
    max_steps=CFG.smoke_test_steps,
)


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [19]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=collator,
)

print("trainer 준비 완료")
print("gpu memory allocated:", round(torch.cuda.memory_allocated() / 1024**3, 3), "GB")
print("gpu memory reserved :", round(torch.cuda.memory_reserved() / 1024**3, 3), "GB")


trainer 준비 완료
gpu memory allocated: 8.779 GB
gpu memory reserved : 11.166 GB


In [20]:
trainer.train()

/opt/conda/lib/python3.11/site-packages/torch/_dynamo/eval_frame.py:929: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/opt/conda/lib/python3.11/site-packages/torch/utils/checkpoint.py:85: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Step,Training Loss,Validation Loss
100,6.687212,6.671240
200,6.651369,6.664217
300,6.686387,6.663083
400,6.651672,6.660969
500,6.684878,6.660074
600,6.622540,6.659371
685,6.651993,6.659039


/opt/conda/lib/python3.11/site-packages/torch/_dynamo/eval_frame.py:929: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/opt/conda/lib/python3.11/site-packages/torch/utils/checkpoint.py:85: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(
/opt/conda/lib/python3.11/site-packages/torch/_dynamo/eval_frame.py:929: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentran

TrainOutput(global_step=685, training_loss=6.957490110745395, metrics={'train_runtime': 8115.3823, 'train_samples_per_second': 0.337, 'train_steps_per_second': 0.084, 'total_flos': 1.066589495144666e+17, 'train_loss': 6.957490110745395, 'epoch': 1.0})

In [21]:
save_dir = os.path.join(CFG.output_dir, "final_adapter")
os.makedirs(save_dir, exist_ok=True)

trainer.save_model(save_dir)
processor.save_pretrained(save_dir)

print("저장 완료:", save_dir)

저장 완료: /workspace/output/final_adapter


In [22]:
import re
from tqdm import tqdm

test_df = pd.read_csv(CFG.test_csv)
print("test 개수:", len(test_df))

def extract_choice(text: str) -> str:
    text = str(text).strip().lower()

    # 가장 먼저 단일 문자 답 추출
    match = re.search(r"\b([abcd])\b", text)
    if match:
        return match.group(1)

    # 혹시 문장으로 나오는 경우 대비
    for ch in ["a", "b", "c", "d"]:
        if ch in text:
            return ch

    # 그래도 없으면 기본값
    return "a"


@torch.no_grad()
def predict_one(row, model, processor):
    image_path = os.path.join(CFG.image_root, row["path"])
    image = Image.open(image_path).convert("RGB")

    user_prompt = build_user_prompt(row)

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": user_prompt},
            ],
        }
    ]

    text = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = processor(
        text=[text],
        images=[image],
        return_tensors="pt",
        padding=True,
    )

    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    generated_ids = model.generate(
        **inputs,
        max_new_tokens=8,
        do_sample=False,
    )

    # 프롬프트 부분 제외하고 생성된 답만 디코딩
    gen_only_ids = generated_ids[:, inputs["input_ids"].shape[1]:]
    output_text = processor.batch_decode(
        gen_only_ids,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=True,
    )[0]

    pred = extract_choice(output_text)
    return pred, output_text

test 개수: 5074


In [23]:
model.eval()

predictions = []
raw_outputs = []

for _, row in tqdm(test_df.iterrows(), total=len(test_df)):
    pred, raw_text = predict_one(row, model, processor)
    predictions.append(pred)
    raw_outputs.append(raw_text)

submission_df = test_df.copy()
submission_df["prediction"] = predictions
submission_df["raw_output"] = raw_outputs

save_path = os.path.join(CFG.output_dir, "test_predictions.csv")
submission_df.to_csv(save_path, index=False, encoding="utf-8-sig")

submit_df = pd.DataFrame({
    "path": test_df["path"],
    "answer": predictions,
})
submit_path = os.path.join(CFG.output_dir, "submission.csv")
submit_df.to_csv(submit_path, index=False, encoding="utf-8-sig")

print("추론 결과 저장 완료:", save_path)
print("제출 파일 저장 완료:", submit_path)
submission_df.head()


  0% 0/5074 [00:00<?, ?it/s]The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
100% 5074/5074 [1:33:07<00:00,  1.10s/it]

추론 결과 저장 완료: /workspace/output/test_predictions.csv
제출 파일 저장 완료: /workspace/output/submission.csv


,id,path,question,a,b,c,d,prediction,raw_output
0,test_0001.jpg,test/test_0001.jpg,사진에 보이는 재활용 가능한 플라스틱 용기는 몇 개입니까?,1개,3개,4개,2개,d,d
1,test_0002.jpg,test/test_0002.jpg,사진에 보이는 재활용 가능한 용기 중 플라스틱 재질인 것은 무엇인가요?,흰색 용기,검은색 용기,알루미늄 용기,투명 비닐 랩,a,a
2,test_0003.jpg,test/test_0003.jpg,사진에 보이는 재활용 가능한 포장재의 재질은 무엇인가요?,유리,금속,플라스틱,종이,c,c
3,test_0004.jpg,test/test_0004.jpg,사진에 보이는 유리병의 뚜껑 색깔은 무엇인가요?,빨간색과 금색,빨간색과 검은색,금색과 검은색,검은색과 흰색,a,a
4,test_0005.jpg,test/test_0005.jpg,책상 위에 있는 재활용품 중 종이 재질로 된 것은 무엇인가요?,플라스틱 뚜껑,커피 컵 홀더,컴퓨터 키보드,노트북,b,b
